# Unidad III – Machine Learning con Enfoque en Aplicaciones

## Clase 2: Construcción de un modelo en un entorno de desarrollo

**Duración:** 3 horas

**Nivel:** Técnico–profesional / certificación

---


## Objetivo general de la clase

Que el estudiante comprenda, diseñe e implemente **un flujo completo de Machine Learning productivo**, desde:

1. Preparación de datos
2. Entrenamiento y evaluación del modelo
3. Persistencia del modelo
4. Exposición del modelo como un **servicio API** reutilizable

Siguiendo **buenas prácticas de ingeniería**, no solo de ciencia de datos.

---


## Competencias a desarrollar

* Comprender el **ciclo de vida real de un modelo ML**
* Diseñar **pipelines reproducibles**
* Separar responsabilidades (datos, modelo, servicio)
* Evaluar modelos con métricas adecuadas
* Desplegar modelos como **servicios desacoplados**

---


---

## Diseño conceptual de un pipeline de Machine Learning

---

## 1. ¿Qué es un pipeline en ML?

Un **pipeline en Machine Learning** es una **estructura de ingeniería** que define, de forma **explícita, ordenada y reproducible**, todas las transformaciones y procesos necesarios para convertir **datos crudos** en **predicciones confiables**, integrando:

* Preprocesamiento de datos
* Ingeniería de características
* Entrenamiento del modelo
* Evaluación
* Inferencia (predicción)

Desde una perspectiva de **ingeniería de software**, un pipeline es un **flujo de procesamiento determinista**, no una simple secuencia de instrucciones.


---

## 1.1 ¿Por qué surge el concepto de pipeline?

El concepto de pipeline nace como respuesta a **fallos estructurales recurrentes** en proyectos de Machine Learning:

### Problemas típicos sin pipeline

| Problema                                 | Impacto                     |
| ---------------------------------------- | --------------------------- |
| Preprocesamiento manual                  | Resultados inconsistentes   |
| Código duplicado                         | Alto costo de mantenimiento |
| Transformaciones distintas en producción | Predicciones erróneas       |
| Falta de trazabilidad                    | Imposible auditar           |
| Modelos no reproducibles                 | Riesgo regulatorio          |

---



## 1.3 Analogía industrial (para comprensión conceptual)

Un pipeline en ML es equivalente a:

* Una **línea de ensamblaje** en manufactura
* Un **ETL** en ingeniería de datos
* Un **pipeline de CI/CD** en DevOps

Cada etapa:

1. Recibe una entrada bien definida
2. Aplica una transformación controlada
3. Entrega una salida estandarizada


---


## 1.4 Componentes fundamentales de un pipeline ML

Un pipeline típico incluye la siguiente arquitectura:

| Capa                | Responsabilidad     |
| ------------------- | ------------------- |
| Data Layer          | Limpieza, selección |
| Feature Engineering | Transformaciones    |
| Model Layer         | Entrenamiento       |
| Evaluation Layer    | Métricas            |
| Service Layer       | Predicción          |


---

## 1.5 ¿Qué garantiza un pipeline?

Un pipeline bien diseñado garantiza:

### 1.6.1 Reproducibilidad

Si se usan los mismos datos y el mismo pipeline, el resultado es el mismo.

Esto es crítico en:

* Finanzas
* Salud
* Auditorías
* Regulación (GDPR, ISO)

---


## 2. Componentes formales de un pipeline ML

```text
Datos → Preprocesamiento → Modelo → Evaluación → Persistencia → Servicio
```

### Cada componente debe ser:

* Independiente
* Reutilizable
* Versionable
* Testeable

---


# Ejercicio

Una bodega quiere estimar la calidad del vino tinto a partir de mediciones físico‑químicas tomadas en laboratorio (acidez, azúcares, pH, alcohol, etc.). Tu tarea es diseñar e implementar un pipeline completo de Machine Learning que permita:
1) Cargar y preparar los datos desde una fuente pública gratuita.
2) Separar características y objetivo evitando data leakage.
3) Entrenar y evaluar un modelo con métricas profesionales.
4) Persistir el pipeline entrenado para reutilización.
5) Exponer un servicio REST con FastAPI para predicciones.

El sistema debe ser:
* Reproducible, profesional y desplegable localmente, usando buenas prácticas de ingeniería
* Uso de scikit-learn Pipeline con preprocesamiento y modelo.
* train_test_split y métricas adecuadas (MAE, MSE, RMSE, R2).
* Persistencia con joblib.
* API /predict con validación de entrada (Pydantic) y manejo básico de errores.

Resultado esperado:
Un proyecto funcional que permita entrenar un modelo, guardarlo y realizar predicciones vía API con datos de entrada en JSON.

Arquitectura por capas (raíz del proyecto)

```
project/
├── data/                # Datos (cache y/o archivos descargados)
├── model/               # Modelo persistido (joblib)
├── pipeline/            # Capa de pipeline
│   └── pipeline.py      # Carga, split y construcción del Pipeline
├── training/            # Capa de entrenamiento
│   └── train.py         # Entrena, evalúa y guarda el modelo
├── service/             # Capa de servicio
│   └── service.py       # Carga y predice con el pipeline
├── api/                 # Capa de API
│   └── api.py           # FastAPI y endpoint /predict
└── requirements.txt     # Dependencias del proyecto
```

---

1) Carga de datos

```pipeline/pipeline.py``` → load_data() descarga o lee el CSV local.

2) Separación de variables

```pipeline/pipeline.py``` → split_features_target() separa `X` y `y` (evita leakage).

4) Construcción del pipeline

`pipeline/pipeline.py` → build_pipeline() crea preprocesamiento + modelo.




In [ ]:
# ===============================
# IMPORTACIONES Y DEPENDENCIAS
# ===============================

from pathlib import Path  
# Pathlib permite manejar rutas de archivos de forma portable y segura
# Evita errores comunes con separadores de ruta entre sistemas operativos

from typing import Tuple  
# Tuple se usa para tipar funciones que retornan múltiples valores
# Mejora legibilidad, mantenibilidad y soporte de herramientas estáticas

import pandas as pd  
# Pandas es la librería estándar para manejo de datos tabulares en ML

from sklearn.compose import ColumnTransformer  
# ColumnTransformer permite aplicar transformaciones distintas
# a diferentes subconjuntos de columnas (buena práctica en pipelines reales)

from sklearn.impute import SimpleImputer  
# SimpleImputer maneja valores faltantes de forma controlada
# Evita errores silenciosos durante entrenamiento e inferencia

from sklearn.linear_model import ElasticNet  
# ElasticNet es un modelo lineal regularizado que combina:
# - L1 (Lasso): selección de variables
# - L2 (Ridge): estabilidad numérica

from sklearn.pipeline import Pipeline  
# Pipeline encapsula todo el flujo de ML:
# preprocesamiento + modelo, garantizando reproducibilidad

from sklearn.preprocessing import StandardScaler  
# StandardScaler estandariza variables numéricas
# Requisito técnico para modelos lineales regularizados


# ===============================
# CONSTANTES GLOBALES
# ===============================

DATA_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "wine-quality/winequality-red.csv"
)
# URL pública del UCI Machine Learning Repository
# Dataset real, ampliamente usado en investigación académica
# Define una constante para evitar valores mágicos en el código


# ===============================
# FUNCIÓN: CARGA Y CACHEO DE DATOS
# ===============================

def load_data(
    data_url: str = DATA_URL,
    cache_path: Path = Path("data/winequality-red.csv"),
) -> pd.DataFrame:
    """
    Descarga el dataset real de calidad de vino y lo cachea localmente.

    Buenas prácticas aplicadas:
    - Evita descargas repetidas
    - Permite ejecución offline
    - Mejora reproducibilidad
    """

    # Verifica si el archivo ya existe localmente
    if cache_path.exists():
        # Si existe, se carga desde disco (más rápido y estable)
        return pd.read_csv(cache_path, sep=";")

    # Si no existe, se descarga desde la URL pública
    df = pd.read_csv(data_url, sep=";")

    # Crea el directorio padre si no existe (evita errores)
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    # Guarda el dataset localmente como CSV
    df.to_csv(cache_path, index=False, sep=";")

    # Retorna el DataFrame cargado
    return df


# ===============================
# FUNCIÓN: SEPARACIÓN FEATURES / TARGET
# ===============================

def split_features_target(
    df: pd.DataFrame,
    target: str = "quality"
) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Separa características (X) y variable objetivo (y).

    Esta separación explícita:
    - Previene data leakage
    - Facilita auditoría
    - Mejora claridad del pipeline
    """

    # X contiene todas las columnas excepto la variable objetivo
    x = df.drop(columns=[target])

    # y contiene únicamente la variable objetivo
    y = df[target]

    # Se retorna una tupla (X, y)
    return x, y


# ===============================
# FUNCIÓN: CONSTRUCCIÓN DEL PIPELINE
# ===============================

def build_pipeline(numeric_features: list[str]) -> Pipeline:
    """
    Construye un pipeline completo de Machine Learning
    con preprocesamiento y modelo integrado.

    Este pipeline es:
    - Reutilizable
    - Serializable
    - Seguro para producción
    """

    # -------------------------------
    # Pipeline para variables numéricas
    # -------------------------------

    numeric_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            # Imputa valores faltantes usando la mediana
            # La mediana es robusta ante outliers

            (
                "scaler",
                StandardScaler()
            ),
            # Escala las variables a media 0 y desviación estándar 1
            # Requisito crítico para ElasticNet
        ]
    )

    # ------------------------------------
    # ColumnTransformer (orquestador)
    # ------------------------------------

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numeric_transformer,
                numeric_features
            )
        ],
        remainder="drop"
        # Todas las columnas no especificadas se descartan
        # Decisión explícita y segura
    )

    # -------------------------------
    # Definición del modelo
    # -------------------------------

    model = ElasticNet(
        alpha=0.1,
        l1_ratio=0.5,
        random_state=42
    )
    # alpha controla la fuerza de regularización
    # l1_ratio equilibra L1 y L2
    # random_state garantiza reproducibilidad

    # -------------------------------
    # Pipeline completo
    # -------------------------------

    pipeline = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", model),
        ]
    )
    # Este pipeline asegura que:
    # - El mismo preprocesamiento se aplique en entrenamiento e inferencia
    # - No exista fuga de datos
    # - El modelo pueda serializarse como una sola unidad

    return pipeline
    # Retorna el pipeline listo para:
    # - Entrenamiento
    # - Evaluación
    # - Despliegue en API


3) Separación train/test

`training/train.py` → train_test_split() divide datos.

5) Entrenamiento

`training/train.py → pipeline.fit()` ajusta con X_train, y_train.

6) Evaluación

`training/train.py → predict()` y métricas MAE/MSE/RMSE/R2.

7) Persistencia

`training/train.py → joblib.dump()` guarda el pipeline en model/.


`training/train.py`



In [ ]:
# ===============================
# IMPORTACIONES Y DEPENDENCIAS
# ===============================

from pathlib import Path  
# Path permite manejar rutas de archivos de forma robusta y multiplataforma
# Es preferible a usar strings para rutas en proyectos profesionales

import joblib  
# joblib se utiliza para serializar (guardar) objetos complejos de Python
# Es el estándar de facto para persistir pipelines y modelos de scikit-learn

import numpy as np  
# NumPy se usa para operaciones numéricas eficientes
# En este script se utiliza principalmente para cálculos de métricas

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
# Métricas estándar de regresión usadas en contextos académicos y empresariales

from sklearn.model_selection import train_test_split  
# Función obligatoria para separar datos de entrenamiento y prueba
# Evita evaluar el modelo con datos que ya vio durante el entrenamiento

from pipeline.pipeline import (
    build_pipeline,
    load_data,
    split_features_target,
)
# Importación de utilidades propias del proyecto:
# - load_data: carga y cachea el dataset
# - split_features_target: separa X e y
# - build_pipeline: construye el pipeline completo de ML


# ===============================
# CONSTANTES DEL PROYECTO
# ===============================

MODEL_PATH = Path("model/pipeline.joblib")  
# Ruta donde se guardará el pipeline entrenado
# Se guarda el pipeline completo, no solo el modelo,
# garantizando consistencia entre entrenamiento e inferencia


# ===============================
# FUNCIÓN DE EVALUACIÓN
# ===============================

def evaluate_regression(
    y_true: np.ndarray,
    y_pred: np.ndarray
) -> dict:
    """
    Calcula métricas profesionales para modelos de regresión.

    Retorna un diccionario para:
    - Facilitar logging
    - Integración con APIs
    - Persistencia o monitoreo
    """

    # Error cuadrático medio (penaliza fuertemente errores grandes)
    mse = mean_squared_error(y_true, y_pred)

    return {
        # Error absoluto medio: interpretación directa en unidades del negocio
        "mae": mean_absolute_error(y_true, y_pred),

        # Error cuadrático medio
        "mse": mse,

        # Raíz del MSE: vuelve a la escala original del problema
        "rmse": np.sqrt(mse),

        # R²: proporción de la varianza explicada por el modelo
        "r2": r2_score(y_true, y_pred),
    }


# ===============================
# FUNCIÓN PRINCIPAL DE ENTRENAMIENTO
# ===============================

def main() -> None:
    """
    Orquesta todo el flujo de entrenamiento del modelo:

    1. Carga de datos
    2. Separación de variables
    3. División train/test
    4. Entrenamiento del pipeline
    5. Evaluación
    6. Persistencia del modelo
    """

    # -------------------------------
    # Carga del dataset
    # -------------------------------

    df = load_data()
    # Se carga el dataset desde caché o desde la URL pública
    # Este paso es determinista y reproducible

    # -------------------------------
    # Separación de features y target
    # -------------------------------

    x, y = split_features_target(df)
    # Se separan explícitamente las variables de entrada (X)
    # y la variable objetivo (y), evitando data leakage

    # -------------------------------
    # División entrenamiento / prueba
    # -------------------------------

    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.2,
        random_state=42,
    )
    # 80% entrenamiento, 20% prueba
    # random_state asegura reproducibilidad de resultados

    # -------------------------------
    # Construcción y entrenamiento del pipeline
    # -------------------------------

    pipeline = build_pipeline(
        numeric_features=list(x.columns)
    )
    # Se construye el pipeline indicando explícitamente
    # qué columnas son numéricas

    pipeline.fit(x_train, y_train)
    # Se entrena TODO el pipeline:
    # - imputación
    # - escalado
    # - modelo
    # en un solo paso coherente

    # -------------------------------
    # Predicción y evaluación
    # -------------------------------

    y_pred = pipeline.predict(x_test)
    # Se generan predicciones sobre datos nunca vistos por el modelo

    metrics = evaluate_regression(y_test, y_pred)
    # Se calculan métricas profesionales de regresión

    # -------------------------------
    # Salida de resultados
    # -------------------------------

    print("Métricas de evaluación")
    print(f"  - MAE: {metrics['mae']:.4f}")
    print(f"  - MSE: {metrics['mse']:.4f}")
    print(f"  - RMSE: {metrics['rmse']:.4f}")
    print(f"  - R2: {metrics['r2']:.4f}")

    # -------------------------------
    # Persistencia del modelo
    # -------------------------------

    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    # Se crea el directorio de destino si no existe

    joblib.dump(pipeline, MODEL_PATH)
    # Se guarda el pipeline completo:
    # - preprocesamiento
    # - modelo
    # Esto es clave para uso posterior en una API

    print(f"Modelo guardado en: {MODEL_PATH}")


# ===============================
# PUNTO DE ENTRADA DEL SCRIPT
# ===============================

if __name__ == "__main__":
    # Esta condición asegura que el entrenamiento
    # solo se ejecute cuando el script se llama directamente
    main()


8) Carga para inferencia

`service/service.py → ModelService.load()` recupera el pipeline.

`service/service.py`


In [ ]:
# ===============================
# IMPORTACIONES Y DEPENDENCIAS
# ===============================

from pathlib import Path  
# Path permite manejar rutas de archivos de forma segura y portable
# Evita errores comunes al trabajar con strings y sistemas operativos distintos

from typing import Optional  
# Optional se usa para indicar que un atributo puede ser None
# Mejora claridad semántica y soporte de herramientas de análisis estático

import joblib  
# joblib es la librería estándar para cargar y guardar pipelines de scikit-learn
# Permite serializar objetos complejos (pipeline completo)

import pandas as pd  
# Pandas se usa para manejar los datos de entrada en formato tabular
# Es el formato esperado por scikit-learn para inferencia

from sklearn.pipeline import Pipeline  
# Se importa el tipo Pipeline únicamente para tipado
# Esto no afecta la ejecución, pero mejora legibilidad y mantenimiento


# ===============================
# CLASE DE SERVICIO DEL MODELO
# ===============================

class ModelService:
    """
    Capa de servicio responsable de:
    - Cargar el pipeline entrenado desde disco
    - Proveer predicciones de forma controlada

    Esta clase desacopla:
    - La lógica de Machine Learning
    - De la capa de API (FastAPI, Flask, etc.)
    """

    def __init__(
        self,
        model_path: str = "model/pipeline.joblib"
    ) -> None:
        """
        Constructor del servicio.

        model_path:
        - Ruta al pipeline entrenado
        - Se define como parámetro para facilitar pruebas y despliegue
        """

        # Se convierte la ruta a objeto Path para manejo seguro
        self.model_path = Path(model_path)

        # El pipeline inicia como None
        # Esto permite controlar explícitamente cuándo se carga el modelo
        self.pipeline: Optional[Pipeline] = None


    # ===============================
    # CARGA DEL MODELO
    # ===============================

    def load(self) -> None:
        """
        Carga el pipeline entrenado desde disco.

        Este método debe ejecutarse:
        - Al iniciar la aplicación
        - Antes de realizar cualquier predicción
        """

        # Verifica que el archivo del modelo exista
        if not self.model_path.exists():
            # Se lanza una excepción clara y explícita
            # Evita fallos silenciosos en producción
            raise FileNotFoundError(
                "No se encontró el modelo entrenado. Ejecuta train.py primero."
            )

        # Se carga el pipeline completo:
        # - Preprocesamiento
        # - Modelo
        self.pipeline = joblib.load(self.model_path)


    # ===============================
    # PREDICCIÓN
    # ===============================

    def predict(self, features: pd.DataFrame) -> pd.Series:
        """
        Realiza predicciones usando el pipeline cargado.

        features:
        - DataFrame con las mismas columnas usadas en entrenamiento
        - No requiere preprocesamiento manual
        """

        # Verifica que el pipeline esté cargado antes de usarlo
        if self.pipeline is None:
            # Esto previene errores difíciles de rastrear en APIs
            raise RuntimeError("El modelo no está cargado.")

        # Se delega la predicción al pipeline
        # El pipeline aplica automáticamente:
        # - imputación
        # - escalado
        # - modelo
        return self.pipeline.predict(features)


9) API de predicción

`api/api.py` → /predict valida JSON, crea DataFrame y llama a service.predict().


In [ ]:
# ===============================
# IMPORTACIONES Y DEPENDENCIAS
# ===============================

from contextlib import asynccontextmanager  
# asynccontextmanager permite definir lógica de inicio (startup)
# y cierre (shutdown) de la aplicación FastAPI de forma limpia

from typing import Dict  
# Dict se usa para tipar explícitamente diccionarios
# Mejora claridad y análisis estático del código

import pandas as pd  
# Pandas se utiliza para construir el DataFrame de entrada
# requerido por el pipeline de scikit-learn

from fastapi import FastAPI, HTTPException  
# FastAPI: framework para construir APIs modernas
# HTTPException: manejo correcto de errores HTTP

from pydantic import BaseModel, Field  
# BaseModel define esquemas de entrada/salida
# Field permite validar reglas de negocio sobre los datos

from service.service import ModelService  
# Capa de servicio desacoplada que maneja el pipeline de ML


# ===============================
# DEFINICIÓN DE FEATURES DEL MODELO
# ===============================

FEATURE_COLUMNS = [
    "fixed acidity",
    "volatile acidity",
    "citric acid",
    "residual sugar",
    "chlorides",
    "free sulfur dioxide",
    "total sulfur dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
]
# Lista que define el ORDEN exacto de columnas esperado por el modelo
# Es crítica para evitar errores silenciosos en inferencia


# ===============================
# MAPEO ENTRE API Y MODELO
# ===============================

FIELD_NAME_MAP: Dict[str, str] = {
    "fixed_acidity": "fixed acidity",
    "volatile_acidity": "volatile acidity",
    "citric_acid": "citric acid",
    "residual_sugar": "residual sugar",
    "chlorides": "chlorides",
    "free_sulfur_dioxide": "free sulfur dioxide",
    "total_sulfur_dioxide": "total sulfur dioxide",
    "density": "density",
    "ph": "pH",
    "sulphates": "sulphates",
    "alcohol": "alcohol",
}
# Este mapeo desacopla:
# - Nombres amigables para la API (snake_case)
# - Nombres originales del dataset/modelo
# Permite cambiar uno sin romper el otro


# ===============================
# ESQUEMA DE ENTRADA (Pydantic)
# ===============================

class WineFeatures(BaseModel):
    """
    Esquema de validación para el endpoint /predict.

    Garantiza:
    - Tipos correctos
    - Reglas de negocio mínimas
    - Datos limpios antes de llegar al modelo
    """

    fixed_acidity: float = Field(..., gt=0)
    volatile_acidity: float = Field(..., gt=0)
    citric_acid: float = Field(..., ge=0)
    residual_sugar: float = Field(..., ge=0)
    chlorides: float = Field(..., ge=0)
    free_sulfur_dioxide: float = Field(..., ge=0)
    total_sulfur_dioxide: float = Field(..., ge=0)
    density: float = Field(..., gt=0)
    ph: float = Field(..., gt=0)
    sulphates: float = Field(..., ge=0)
    alcohol: float = Field(..., gt=0)
    # Estas validaciones:
    # - Previenen entradas absurdas
    # - Protegen al modelo
    # - Mejoran confiabilidad de la API


# ===============================
# INSTANCIA DEL SERVICIO DE MODELO
# ===============================

service = ModelService()
# Se crea una única instancia del servicio
# El pipeline se carga una sola vez al iniciar la aplicación


# ===============================
# CICLO DE VIDA DE LA APLICACIÓN
# ===============================

@asynccontextmanager
async def lifespan(app: FastAPI):
    """
    Maneja el ciclo de vida de la aplicación.

    Se ejecuta:
    - Antes de aceptar peticiones (startup)
    - Al cerrar la aplicación (shutdown)
    """

    try:
        service.load()
        # Carga el pipeline entrenado al iniciar la API
        # Evita cargar el modelo en cada request
    except FileNotFoundError:
        service.pipeline = None
        # Permite que la API levante aunque el modelo no exista
        # Ideal para entornos de desarrollo o despliegue progresivo

    yield
    # Aquí podría agregarse lógica de cierre si fuera necesario


# ===============================
# CREACIÓN DE LA APLICACIÓN FASTAPI
# ===============================

app = FastAPI(
    title="Wine Quality Predictor",
    version="1.0.0",
    lifespan=lifespan,
)
# Se asocia explícitamente el lifespan
# Esto es una práctica moderna recomendada por FastAPI


# ===============================
# ENDPOINT DE PREDICCIÓN
# ===============================

@app.post("/predict")
def predict(features: WineFeatures) -> dict:
    """
    Endpoint que recibe características del vino
    y retorna la calidad estimada por el modelo.
    """

    # Verifica que el modelo esté cargado
    if service.pipeline is None:
        raise HTTPException(
            status_code=503,
            detail="Modelo no cargado.",
        )
        # 503 indica que el servicio no está disponible temporalmente

    # Convierte el objeto Pydantic a diccionario
    payload = features.model_dump()

    # Mapea los nombres de la API a los nombres del modelo
    row = {
        FIELD_NAME_MAP[key]: value
        for key, value in payload.items()
    }

    # Construye el DataFrame con el orden correcto
    dataframe = pd.DataFrame(
        [row],
        columns=FEATURE_COLUMNS,
    )
    # Usar DataFrame garantiza compatibilidad total con scikit-learn

    try:
        prediction = service.predict(dataframe)[0]
        # Se obtiene la predicción única
    except ValueError as exc:
        # Captura errores típicos de entrada inválida
        raise HTTPException(
            status_code=400,
            detail=str(exc),
        ) from exc

    # Retorna la predicción como JSON
    return {
        "predicted_quality": round(float(prediction), 3)
    }


---

`requirements.txt`

In [ ]:
fastapi
uvicorn[standard]
pandas
scikit-learn
joblib


---


## Para ejecutar la app desde la raíz del proyecto:

*1) Instala dependencias:*

`pip install -r requirements.txt`

*2) Entrena y guarda el modelo:*

`python -m training.train`

*3) Levanta la API:*

`uvicorn api.api:app --reload`

*4) Prueba el endpoint /predict:*

> Aquí hay 3 ejemplos listos para probar la API.

**Ejemplo 1 (vino típico)**

```
{
  "fixed_acidity": 7.4,
  "volatile_acidity": 0.7,
  "citric_acid": 0.0,
  "residual_sugar": 1.9,
  "chlorides": 0.076,
  "free_sulfur_dioxide": 11,
  "total_sulfur_dioxide": 34,
  "density": 0.9978,
  "ph": 3.51,
  "sulphates": 0.56,
  "alcohol": 9.4
}
```

**Ejemplo 2 (más ácido y alcohólico)**

```
{
  "fixed_acidity": 9.2,
  "volatile_acidity": 0.52,
  "citric_acid": 0.24,
  "residual_sugar": 2.2,
  "chlorides": 0.088,
  "free_sulfur_dioxide": 15,
  "total_sulfur_dioxide": 60,
  "density": 0.9981,
  "ph": 3.32,
  "sulphates": 0.62,
  "alcohol": 11.2
}
```

**Ejemplo 3 (más suave y menor acidez volátil)**

```
{
  "fixed_acidity": 6.8,
  "volatile_acidity": 0.38,
  "citric_acid": 0.18,
  "residual_sugar": 2.0,
  "chlorides": 0.068,
  "free_sulfur_dioxide": 22,
  "total_sulfur_dioxide": 38,
  "density": 0.9965,
  "ph": 3.45,
  "sulphates": 0.54,
  "alcohol": 10.1
}
```

**Explicación de cada dato**

* fixed_acidity: acidez fija (no volátil) del vino.
* volatile_acidity: acidez volátil (ácidos que se evaporan).
* citric_acid: ácido cítrico presente.
* residual_sugar: azúcar residual después de fermentar.
* chlorides: sales (cloruros).
* free_sulfur_dioxide: dióxido de azufre libre (conservación).
* total_sulfur_dioxide: dióxido de azufre total.
* density: densidad del vino.
* ph: nivel de acidez (pH).
* sulphates: sulfatos (afectan sabor y conservación).
* alcohol: porcentaje de alcohol.

